# S06 — Optimizers

**Week 4 · Mon Sep 14, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s06_optimizers.ipynb)

Every cell below is a worked example from the [S06 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s06/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s06.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s06.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Nesterov momentum: looking before you leap


*Expected output starts with:* `  loss after step:          10          30          50         100         200`


In [ ]:
import torch

torch.manual_seed(0)

# The same bent bowl as the from-scratch race, now through torch.optim.SGD,
# comparing heavy-ball momentum against its Nesterov variant.
curv = torch.tensor([1.0, 25.0])

def loss_fn(w):
    return 0.5 * (curv * w * w).sum()

def run(nesterov):
    w = torch.nn.Parameter(torch.tensor([2.0, 2.0]))
    opt = torch.optim.SGD([w], lr=0.03, momentum=0.9, nesterov=nesterov)
    out = {}
    for t in range(1, 201):
        opt.zero_grad()
        loss = loss_fn(w)
        loss.backward()
        opt.step()
        if t in (10, 30, 50, 100, 200):
            out[t] = loss_fn(w).item()
    return out

checkpoints = (10, 30, 50, 100, 200)
print(f"{'loss after step:':>18s}" + "".join(f"{t:>12d}" for t in checkpoints))
for name, nesterov in [("heavy-ball", False), ("Nesterov", True)]:
    h = run(nesterov)
    print(f"{name:>18s}" + "".join(f"{h[t]:12.6f}" for t in checkpoints))

## Racing the update rules on a bent bowl


*Expected output starts with:* `start loss = 52.0000,  lr = 0.03`


In [ ]:
import numpy as np

np.random.seed(0)

# An ill-conditioned quadratic bowl: steep in w[1], shallow in w[0].
# L(w) = 0.5 * (1 * w[0]^2 + 25 * w[1]^2), minimum at (0, 0).
curv = np.array([1.0, 25.0])

def loss(w):
    return 0.5 * np.sum(curv * w * w)

def grad(w):
    return curv * w

w0 = np.array([2.0, 2.0])
steps = 200
checkpoints = (10, 50, 100, 200)

def run(update, state_init):
    w, state = w0.copy(), state_init()
    history = {}
    for t in range(1, steps + 1):
        g = grad(w)
        w, state = update(w, g, state, t)
        if t in checkpoints:
            history[t] = loss(w)
    return history

lr = 0.03

def sgd(w, g, s, t):
    return w - lr * g, s

def momentum(w, g, s, t):           # heavy-ball, beta = 0.9
    v = 0.9 * s["v"] + g
    return w - lr * v, {"v": v}

def rmsprop(w, g, s, t):            # beta2 = 0.99
    r = 0.99 * s["r"] + 0.01 * g * g
    return w - lr * g / (np.sqrt(r) + 1e-8), {"r": r}

def adam(w, g, s, t):               # beta1 = 0.9, beta2 = 0.999
    m = 0.9 * s["m"] + 0.1 * g
    v = 0.999 * s["v"] + 0.001 * g * g
    m_hat = m / (1 - 0.9 ** t)      # bias correction
    v_hat = v / (1 - 0.999 ** t)
    return w - lr * m_hat / (np.sqrt(v_hat) + 1e-8), {"m": m, "v": v}

print(f"start loss = {loss(w0):.4f},  lr = {lr}")
print(f"{'loss after step:':>16s}" + "".join(f"{t:>12d}" for t in checkpoints))
for name, update, init in [
    ("SGD", sgd, dict),
    ("momentum", momentum, lambda: {"v": 0.0}),
    ("RMSProp", rmsprop, lambda: {"r": 0.0}),
    ("Adam", adam, lambda: {"m": 0.0, "v": 0.0}),
]:
    h = run(update, init)
    print(f"{name:>16s}" + "".join(f"{h[t]:12.6f}" for t in checkpoints))

## SGD versus Adam on a real network


*Expected output starts with:* `loss at epoch:        50       100       200       400     acc`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Synthetic "ring" classification: class 1 if the point lies in an annulus.
n = 600
X = torch.randn(n, 2) * 1.5
r = X.norm(dim=1)
y = ((r > 1.0) & (r < 2.5)).long()

def make_model():
    torch.manual_seed(1)                     # identical init for every optimizer
    return nn.Sequential(
        nn.Linear(2, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 2),
    )

def train(opt_name):
    model = make_model()
    if opt_name == "SGD":
        opt = torch.optim.SGD(model.parameters(), lr=0.1)
    elif opt_name == "SGD+mom":
        opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    checkpoints = {}
    for epoch in range(1, 401):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        opt.step()
        if epoch in (50, 100, 200, 400):
            checkpoints[epoch] = loss.item()
    with torch.no_grad():
        acc = (model(X).argmax(dim=1) == y).float().mean().item()
    return checkpoints, acc

print(f"{'loss at epoch:':>14s}{50:>10d}{100:>10d}{200:>10d}{400:>10d}{'acc':>8s}")
for name in ("SGD", "SGD+mom", "Adam"):
    cp, acc = train(name)
    row = "".join(f"{cp[e]:10.4f}" for e in (50, 100, 200, 400))
    print(f"{name:>14s}{row}{acc:8.3f}")

## AdamW: weight decay done correctly


*Expected output starts with:* `after 500 steps (started at 1.0, weight_decay=0.1):`


In [ ]:
import torch

torch.manual_seed(0)

# Two weights, both starting at 1.0.
# Each step, the "data gradient" is +/-30 for w_a but +/-0.3 for w_b
# (alternating sign, so neither weight is pulled anywhere on average).
# The only systematic force is weight decay 0.1. Where does each weight end up?

def run(optimizer_cls):
    w = torch.nn.Parameter(torch.tensor([1.0, 1.0]))
    opt = optimizer_cls([w], lr=0.01, weight_decay=0.1)
    for t in range(500):
        sign = 1.0 if t % 2 == 0 else -1.0
        w.grad = torch.tensor([30.0 * sign, 0.3 * sign])  # loss gradient only
        opt.step()
        opt.zero_grad()
    return w.detach()

adam = run(torch.optim.Adam)     # L2 folded into the gradient
adamw = run(torch.optim.AdamW)   # decay applied directly to the weight

print("after 500 steps (started at 1.0, weight_decay=0.1):")
print(f"  Adam  (L2-in-gradient):  w_a = {adam[0]:.4f}   w_b = {adam[1]:.4f}")
print(f"  AdamW (decoupled decay): w_a = {adamw[0]:.4f}   w_b = {adamw[1]:.4f}")

## Learning-rate schedules


*Expected output starts with:* `step    0: lr = 0.000000`


In [ ]:
import math
import torch

torch.manual_seed(0)

model = torch.nn.Linear(4, 1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

total_steps, warmup_steps = 1000, 100

def lr_lambda(step):
    if step < warmup_steps:                       # linear warmup: 0 -> 1
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))   # cosine: 1 -> 0

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

for step in range(total_steps):
    # ... forward, loss.backward(), opt.step() would go here ...
    if step in (0, 50, 100, 500, 900, 999):
        print(f"step {step:4d}: lr = {sched.get_last_lr()[0]:.6f}")
    opt.step()          # scheduler warns if step() precedes it
    sched.step()

## Does warmup actually do anything?


*Expected output starts with:* ` base lr |       worst loss, steps 1-50 |       final accuracy`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Ring data, as in the SGD-versus-Adam experiment.
n = 600
X = torch.randn(n, 2) * 1.5
r = X.norm(dim=1)
y = ((r > 1.0) & (r < 2.5)).long()

def make_model():
    torch.manual_seed(1)                     # identical init for every run
    return nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 2),
    )

def train(base_lr, warmup_steps, total_steps=600, batch=32):
    """Minibatch Adam; returns the worst loss seen in the first 50 steps
    and the final full-data accuracy."""
    model = make_model()
    opt = torch.optim.Adam(model.parameters(), lr=base_lr)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(2)     # identical batch order for every run
    step, worst_early = 0, 0.0
    while step < total_steps:
        for idx in torch.randperm(n, generator=g).split(batch):
            step += 1
            if step > total_steps:
                break
            lr = base_lr * min(1.0, step / warmup_steps) if warmup_steps else base_lr
            for group in opt.param_groups:
                group["lr"] = lr
            opt.zero_grad()
            loss = loss_fn(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
            worst_early = max(worst_early, loss.item()) if step <= 50 else worst_early
    model.eval()
    with torch.no_grad():
        acc = (model(X).argmax(dim=1) == y).float().mean().item()
    return worst_early, acc

print(f"{'base lr':>8s} | {'worst loss, steps 1-50':>28s} | {'final accuracy':>20s}")
print(f"{'':>8s} | {'no warmup':>13s}{'warmup 150':>13s}  | {'no warmup':>9s}{'warmup 150':>11s}")
for base_lr in (0.01, 0.05, 0.1):
    w0, a0 = train(base_lr, warmup_steps=0)
    w1, a1 = train(base_lr, warmup_steps=150)
    print(f"{base_lr:8.2f} | {w0:13.4f}{w1:13.4f}  | {a0:9.3f}{a1:11.3f}")

## A schedule bake-off


*Expected output starts with:* ` full-data loss at step:       200       400       800`


In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(0)

# Ring data again; four learning-rate schedules, same model, same base lr.
n = 600
X = torch.randn(n, 2) * 1.5
r = X.norm(dim=1)
y = ((r > 1.0) & (r < 2.5)).long()

total_steps, base_lr, warmup = 800, 1e-2, 80

def schedule(name, step):
    """Multiplier on base_lr at a given step (0-indexed)."""
    if name == "constant":
        return 1.0
    if name == "step decay":            # x0.1 at 50% and again at 75%
        if step < total_steps // 2:
            return 1.0
        return 0.1 if step < 3 * total_steps // 4 else 0.01
    if name == "cosine":
        return 0.5 * (1 + math.cos(math.pi * step / total_steps))
    if name == "warmup+cosine":
        if step < warmup:
            return step / warmup
        prog = (step - warmup) / (total_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))

def train(name, batch=32):
    torch.manual_seed(1)
    model = nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 2),
    )
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(2)
    step, checkpoints = 0, {}
    while step < total_steps:
        for idx in torch.randperm(n, generator=g).split(batch):
            if step >= total_steps:
                break
            for group in opt.param_groups:
                group["lr"] = base_lr * schedule(name, step)
            opt.zero_grad()
            loss = loss_fn(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
            step += 1
            if step in (200, 400, 800):
                model.eval()
                with torch.no_grad():
                    checkpoints[step] = loss_fn(model(X), y).item()
                model.train()
    return checkpoints

print(f"{'full-data loss at step:':>24s}{200:>10d}{400:>10d}{800:>10d}")
for name in ("constant", "step decay", "cosine", "warmup+cosine"):
    cp = train(name)
    print(f"{name:>24s}" + "".join(f"{cp[s]:10.4f}" for s in (200, 400, 800)))

## Try it yourself

1. In the bowl experiment, raise the learning rate to `0.075` and rerun. Which methods diverge, which merely oscillate, and why does the steep-direction stability bound `2/25 = 0.08` predict what you see?
2. Add Nesterov momentum to the from-scratch NumPy bowl race (evaluate the gradient at `w - lr * beta * v` before updating `v`), and compare your step-10 loss against both rows of the torch-based Nesterov table above. (Your numbers will not match torch exactly — its `nesterov=True` uses a slightly different but equivalent-in-spirit reformulation; explain the discrepancy you see.)
3. In the ring experiment, sweep Adam's learning rate over `{1e-4, 3e-4, 1e-3, 3e-3, 1e-2}` and find where it beats SGD+momentum at `lr = 0.1`. What does this tell you about the earlier table?
4. Reproduce the AdamW demonstration with `beta1 = 0` (no momentum). Predict the final `w_a` and `w_b` for both optimizers before running.


---

Full discussion of everything above: [S06 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s06/).
